In [1]:
%load_ext autoreload
%autoreload 2


import os
import logging
import pandas as pd
import numpy as np
import time
import torch
import wandb
import copy
from tqdm import tqdm


from src.constants import DEVICE
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler
from src.dataset.load_dataset import load_dataset, generate_K_fold_cross_validation_splits
from src.models.tools.get_classification_model import get_classification_model
from src.dataset.get_dataloader import make_dataloader   
from src.dataset.get_transforms import get_transforms
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler



/home/data/.predRTenv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/data/.predRTenv/lib64/python3.9/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
from src.config_presets.tools.get_config import get_config

config = get_config('Trial32_Config_VM')




src/config_presets/Base_config.yaml
src/config_presets/Trial32_Config_VM.yaml


In [3]:
# load the data
df_train_val, df_test = load_dataset(config)
    
dataset_split_dict = generate_K_fold_cross_validation_splits(config, df_train_val)

# cap the number of iterations, if it is less than the number of k-splits to make
n_iterations = config['data']['kFolds']['n_iterations']

# get the data transforms
train_transforms, val_transforms = get_transforms(config)
# get the loss function
loss_function = get_loss_function(config)

metricHandler = mainMetricHandler(config)



Removed patients (no image data) = 0
Train/Val dataset 1092 (79.53386744355426%), Test dataset 281 (20.46613255644574%)


/home/data/.predRTenv/lib64/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [4]:
train_data, val_data = dataset_split_dict[0]['train'], dataset_split_dict[0]['val']

train_loader, metadata = make_dataloader(config, train_data, train_transforms, validation_mode=False)
val_loader, _ = make_dataloader(config, val_data, val_transforms, validation_mode=True)



In [5]:
import torch

torch.cuda.empty_cache()

In [32]:
%load_ext autoreload
%autoreload 2


from src.models.tools.get_classification_model import get_classification_model

config['model']['model_name'] = 'TransRP'
config['general']['experiment_name'] = "modulated_TransRP"
config['model']['linear_units'] = []
config['general']['trialNumber'] = "protons_in_densenet"

config['model']['modulated_backbone'] = True


config['columns']['clinical_features'] = [
                      #'Geslacht', 'Leeftijd',
                      #"Concurrent_Chemo_or_Bioradiation", 
                      "Protons",
                      #'P16_Negatief', 'P16_Niet_bepaald', 'P16_Positief', 
                      #'Roken_Ja, in verleden', 'Roken_Ja, nog steeds', 'Roken_Nee',

                      'Aspiration_W01_Helemaal_niet','Aspiration_W01_Een_beetje',  'Aspiration_W01_Nogal_Heel_erg', 
                      'Dysphagia_W01_Grade0_1', 'Dysphagia_W01_Grade2', 'Dysphagia_W01_Grade3_4', 
                      'Sticky_W01_Helemaal_niet', 'Sticky_W01_Een_beetje', 'Sticky_W01_Nogal_Heel_erg',
                      'Taste_W01_Helemaal_niet','Taste_W01_Een_beetje',  'Taste_W01_Nogal_Heel_erg', 
                      'Xerostomia_W01_Helemaal_niet', 'Xerostomia_W01_Een_beetje', 'Xerostomia_W01_Nogal_Heel_erg', 
                     # 
                       
                    #   'WHO_0', 'WHO_1', 'WHO_2', 'WHO_3', 'WHO_4', 
                    #   'T0', 'T1', 'T2', 'T3', 'T4', 'Tis', 'Tx', 
                    #   'N0', 'N1', 'N2', 'N3', 
                    #   
                      ]

config['general']['resultsCurrentDirectory'] = os.path.join(os.getcwd(), 'test_TRP_CLS2')


config['model']['TransRP']['clinical_features_method'] = 'cls'

# config['model']['TransRP']['vit_dim'] = 128
# config['model']['TransRP']['cls_gating'] = False

# config["model"]["TransRP"]["cls_merge_image_features"] = False
# config["model"]["TransRP"]["cls_per_class_weighting"] = True


config['model']['TransRP']['image_encoder'] = 'modulated_densenet'
#
config['model']['convnext']['model_size'] = 'base'
config['model']['convnext']['kernel_size'] = 3
config['model']['convnext']['patch_size'] = 4


# config['model']['model_name'] = 'densenet'

# config['general']['resultsCurrentDirectory'] = os.path.join(os.getcwd(), 'test_densenet121')


config['model']['linear_units'] = []

config['training']['batch_size'] = 4

model = get_classification_model(config, metadata=metadata, save_summary=True)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Modulated backbone:  True
17
hello 1 tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
       device='cuda:0')
HELLO tensor([[0.4090, 0.4510, 0.8753, 0.4332, 0.6142, 0.7491, 0.0122, 0.0426, 0.6730,
         0.8628, 0.1489, 0.7785, 0.4622, 0.6565, 0.8478, 0.3568, 0.9457],
        [0.3388, 0.5518, 0.7144, 0.7997, 0.3465, 0.2168, 0.5473, 0.6954, 0.8530,
         0.7215, 0.3381, 0.1244, 0.6602, 0.7177, 0.7883, 0.0708, 0.3364],
        [0.0389, 0.3922, 0.3824, 0.4803, 0.4738, 0.6237, 0.5587, 0.7757, 0.7025,
         0.0510, 0.6235, 0.0026, 0.1571, 0.6631, 0.3413, 0.2540, 0.3079],
        [0.6585, 0.9814, 0.2064, 0.0380, 0.7375, 0.1165, 0.2426, 0.9866, 0.4941,
         0.1366, 0.7805, 0.1081, 0.6432, 0.6442, 0.7057, 0.5879, 0.8829]],
       device='cuda:0')
hello 1 tensor([[0.4090, 0.4510, 0.8753, 0.4332, 0.6142, 0.7491, 0.0122, 0.0426, 0.6730,
         0.8628, 0.1489, 0.7785, 0.4

In [2]:
string = "P16_Negatief;P16_Niet_bepaald;P16_Positief;WHO_0;WHO_1;WHO_2;WHO_3;WHO_4;T0;T1;T2;T3;T4;Tis;Tx;N0;N1;N2;N3;Roken_Ja, in verleden;Roken_Ja, nog steeds;Roken_Nee"

string = string.split(';')
print(string)


['P16_Negatief', 'P16_Niet_bepaald', 'P16_Positief', 'WHO_0', 'WHO_1', 'WHO_2', 'WHO_3', 'WHO_4', 'T0', 'T1', 'T2', 'T3', 'T4', 'Tis', 'Tx', 'N0', 'N1', 'N2', 'N3', 'Roken_Ja, in verleden', 'Roken_Ja, nog steeds', 'Roken_Nee']


In [25]:
config['model']['output_head']

{'name': 'multitox'}

In [17]:
config['model']['linear_units_endpoint']

[32]

In [ ]:

for name, module in model.encoder.named_modules():
    last_module = module
print(last_module)

AdaptiveAvgPool3d(output_size=(1, 1, 1))


In [ ]:
model.output_head.linear_layers.shared_fc_layers.[4]

SyntaxError: invalid syntax (2619673146.py, line 1)

In [ ]:
getattr(model.output_head.linear_layers.shared_fc_layers, torch.nn.Linear)

TypeError: getattr(): attribute name must be string

In [ ]:
model.output_head.linear_layers.shared_fc_layers
last_linear_layer = None
last_name = None
for name, layer in model.output_head.linear_layers.shared_fc_layers.named_modules():
    if isinstance(layer, (torch.nn.Linear, torch.nn.LazyLinear)):
        last_linear_layer = layer
        last_name = name

print(last_name)
print(last_linear_layer)

Linear_shared_2
Linear(in_features=128, out_features=64, bias=True)


In [ ]:
model.output_head.linear_layers.shared_fc_layers.name

AttributeError: 'ModuleList' object has no attribute 'name'

In [ ]:
getattr(model.output_head.transformer.layers, 'Attention Block 3')[1]

In [ ]:
model.output_head.linear_layers.shared_fc_layers


In [ ]:
def find_last_module(module, target_name='endpoint_heads'):
    last_module = None
    for name, submodule in module.named_children():
        if target_name in name:
            break
        last_module = submodule
        
        # Recursively search in submodules
        found_module = find_last_module(submodule, target_name)
        if found_module:
            last_module = found_module
            print(name)
    return last_module

last_module = find_last_module(model)
print(last_module)

0
block0
downsample
0
block1
downsample
0
block2
downsample
0
block3
backbone
encoder
MultiToxOutputHead(
  (clinical_variables_shared_fc_layers): ModuleList()
  (shared_fc_layers): ModuleList()
  (endpoint_heads): ModuleDict(
    (Aspiration_M06): ModuleList(
      (0): Dropout(p=0.1, inplace=False)
      (1): Linear(in_features=273, out_features=64, bias=True)
      (2): LeakyReLU(negative_slope=0.1)
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=64, out_features=1, bias=True)
    )
    (Dysphagia_M06): ModuleList(
      (0): Dropout(p=0.1, inplace=False)
      (1): Linear(in_features=273, out_features=64, bias=True)
      (2): LeakyReLU(negative_slope=0.1)
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=64, out_features=1, bias=True)
    )
    (Sticky_M06): ModuleList(
      (0): Dropout(p=0.1, inplace=False)
      (1): Linear(in_features=273, out_features=64, bias=True)
      (2): LeakyReLU(negative_slope=0.1)
      (3): Dropout(p=0.1,

In [ ]:
for name, module in model.output_head.named_children():
    print(name)

patch_embedding
transformer
norm
linear_layers
clc_embed


In [ ]:
model.output_head.shared_fc_layers

ModuleList()

In [ ]:


for name, module in model.output_head.transformer.layers.named_modules():
    print(name)


Attention Block 0
Attention Block 0.0
Attention Block 0.0.norm
Attention Block 0.0.attend
Attention Block 0.0.dropout
Attention Block 0.0.to_qkv
Attention Block 0.0.to_out
Attention Block 0.0.to_out.0
Attention Block 0.0.to_out.1
Attention Block 0.1
Attention Block 0.1.net
Attention Block 0.1.net.0
Attention Block 0.1.net.1
Attention Block 0.1.net.2
Attention Block 0.1.net.3
Attention Block 0.1.net.4
Attention Block 0.1.net.5
Attention Block 1
Attention Block 1.0
Attention Block 1.0.norm
Attention Block 1.0.attend
Attention Block 1.0.dropout
Attention Block 1.0.to_qkv
Attention Block 1.0.to_out
Attention Block 1.0.to_out.0
Attention Block 1.0.to_out.1
Attention Block 1.1
Attention Block 1.1.net
Attention Block 1.1.net.0
Attention Block 1.1.net.1
Attention Block 1.1.net.2
Attention Block 1.1.net.3
Attention Block 1.1.net.4
Attention Block 1.1.net.5
Attention Block 2
Attention Block 2.0
Attention Block 2.0.norm
Attention Block 2.0.attend
Attention Block 2.0.dropout
Attention Block 2.0.t

In [ ]:
att_layer_name = f"Attention Block {config['model']['TransRP']['vit_depth'] - 1}"

In [ ]:
for name, layer in model.named_modules():
    #if isinstance(layer, nn.ReLU):
    print(name, layer)

    pytorch_layer_obj = getattr(model, name)

 MultiTox_Classifier(
  (encoder): DenseNet(
    (features): Sequential(
      (conv0): Conv3d(3, 64, kernel_size=(5, 5, 5), stride=(2, 2, 2), padding=(2, 2, 2), bias=False)
      (norm0): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): LeakyReLU(negative_slope=0)
      (pool0): MaxPool3d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock_1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (layers): Sequential(
            (norm1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu1): LeakyReLU(negative_slope=0)
            (conv1): Conv3d(64, 128, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
            (norm2): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu2): LeakyReLU(negative_slope=0)
            (conv2): Conv3d(128, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=Fa

AttributeError: 'MultiTox_Classifier' object has no attribute ''

In [ ]:
model.output_head.transformer.layers[]

TypeError: 'str' object cannot be interpreted as an integer

In [ ]:
getattr(model.output_head.transformer.layers, 'Attention Block 1')

ModuleList(
  (0): Attention(
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (attend): Softmax(dim=-1)
    (dropout): Dropout(p=0.2, inplace=False)
    (to_qkv): Linear(in_features=128, out_features=384, bias=False)
    (to_out): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): Dropout(p=0.2, inplace=False)
    )
  )
  (1): FeedForward(
    (net): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=512, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=512, out_features=128, bias=True)
      (5): Dropout(p=0.2, inplace=False)
    )
  )
)

In [ ]:
attention_block = getattr(model.output_head.transformer.layers, 'Attention Block 3')
print(attention_block)

attention_block[1]

ModuleList(
  (0): Attention(
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (attend): Softmax(dim=-1)
    (dropout): Dropout(p=0.2, inplace=False)
    (to_qkv): Linear(in_features=128, out_features=384, bias=False)
    (to_out): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): Dropout(p=0.2, inplace=False)
    )
  )
  (1): FeedForward(
    (net): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=512, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=512, out_features=128, bias=True)
      (5): Dropout(p=0.2, inplace=False)
    )
  )
)


FeedForward(
  (net): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=512, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): Dropout(p=0.2, inplace=False)
  )
)

In [ ]:
model.output_head.linear_layers[-1]

TypeError: 'MultiToxOutputHead' object is not subscriptable

In [ ]:
model.output_head.transformer.layers.[0].feed_forward.parameters()

SyntaxError: invalid syntax (3408966411.py, line 1)